# Screening pa reemplazar a PYPL
El profe pidió cambiar PYPL en la asesoría del 09/11. Acá miro cuáles acciones del Nasdaq-100 van bien este año y han crecido parejo.

In [3]:
import numpy as np, pandas as pd, yfinance as yf

# --- 1. lista del nasdaq-100 ---
# la copié de investing.com (fuente del profe), consultada el 10/11/2026
comp = pd.read_csv("../data/raw/nasdaq100_componentes_investing.csv")
tickers = comp["ticker"].str.replace(".", "-", regex=False).tolist()  # yahoo usa guion
print(len(tickers), "tickers")   # tienen que salir 101

# --- 2. bajo los mismos 10 años que ya tengo en la bitácora ---
px = yf.download(tickers, start="2016-10-31", end="2026-10-31",
                 auto_adjust=False, progress=False)["Adj Close"]
print("última fecha:", px.index[-1].date())   # tiene que dar 2026-10-30

# las que no tienen precio desde el primer día no me sirven (historia incompleta)
antes = px.shape[1]
px = px.loc[:, px.iloc[0].notna()]
print(f"quedan {px.shape[1]} de {antes} con los 10 años completos")

ret = np.log(px).diff().iloc[1:]   # retornos log diarios
corr = ret.corr()

# --- 3. métricas pa comparar ---
cierre_2025 = px.loc[:"2025-12-31"].iloc[-1]                          # último precio del 2025
anual = px.groupby(px.index.year).last().pct_change().loc[2017:2025]  # solo años completos

# R2 de log(precio) contra el tiempo: si da cerca de 1, la acción sube parejo
r2 = np.log(px).apply(lambda s: np.corrcoef(np.arange(len(s.dropna())), s.dropna())[0, 1] ** 2)

tabla = pd.DataFrame({
    "ytd_2026":  px.iloc[-1] / cierre_2025 - 1,      # lo que lleva este año
    "ret_anual": np.exp(ret.mean() * 252) - 1,       # igual que en la bitácora
    "vol_anual": ret.std() * np.sqrt(252),
    "anos_pos":  (anual > 0).sum(),                  # de 9 años, cuántos cerró en verde
    "r2_tend":   r2,
    "max_dd":    (px / px.cummax() - 1).min(),       # la peor caída desde un pico
    "corr_CSCO": corr["CSCO"],
    "corr_ADI":  corr["ADI"],
})
tabla["sharpe"] = (tabla["ret_anual"] - 0.04929) / tabla["vol_anual"]  # rf de la bitácora

# le pego el nombre de la empresa pa que sea más fácil leer la tabla
tabla = tabla.join(comp.set_index("ticker")["empresa"])

# --- 4. filtro y resultado ---
# filtro suave: que vaya ganando este año y que haya cerrado en verde la mayoría de años
cand = (tabla.drop(["CSCO", "ADI", "PYPL"], errors="ignore")
             .query("ytd_2026 > 0 and anos_pos >= 6")
             .sort_values("ytd_2026", ascending=False))

cand.to_csv("../data/processed/screening_reemplazo_pypl.csv")
print(len(cand), "candidatas pasaron el filtro")
cand.head(15).round(3)

101 tickers


c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quo

última fecha: 2026-10-30
quedan 83 de 101 con los 10 años completos
46 candidatas pasaron el filtro


,ytd_2026,ret_anual,vol_anual,anos_pos,r2_tend,max_dd,corr_CSCO,corr_ADI,sharpe,empresa
Ticker,,,,,,,,,,
MU,2.102,0.489,0.518,6,0.678,-0.576,0.403,0.616,0.849,Micron
AMD,1.357,0.531,0.569,6,0.881,-0.654,0.377,0.545,0.846,AMD
STX,1.348,0.397,0.438,7,0.714,-0.570,0.381,0.481,0.793,Seagate
MRVL,1.315,0.322,0.537,6,0.855,-0.619,0.443,0.620,0.507,Marvell
LITE,1.281,0.381,0.593,7,0.337,-0.669,0.378,0.496,0.559,Lumentum Holdings Inc
WDC,1.127,0.251,0.509,6,0.207,-0.705,0.400,0.552,0.397,Western Digital
PANW,0.986,0.305,0.404,8,0.956,-0.480,0.341,0.304,0.635,Palo Alto Networks
INTC,0.782,0.088,0.474,6,0.001,-0.708,0.397,0.532,0.083,Intel
TXN,0.772,0.189,0.318,6,0.876,-0.334,0.504,0.830,0.438,Texas Instruments
